In [12]:

#====================================================================================
# Imports
#====================================================================================

from __future__ import annotations
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any, Dict, List, Optional, Union
from urllib.parse import urlparse, parse_qs, urlunparse
from datetime import date

import requests
import polars as pl
import json


In [13]:
#====================================================================================
# Variables
#====================================================================================
current_season = 20252026
overlap_duration = 35 #seconds

In [ ]:

# data tables by season Id - will be easy to pull and can presuably just pull one time for historical seasons. 
team_codes_csv = "https://api.nhle.com/stats/rest/en/team" #static saved as csv
games = "https://api.nhle.com/stats/rest/en/game/summary?&cayenneExp=season=20232024" #teams, dates, scores
skater_summary = "https://api.nhle.com/stats/rest/en/skater/summary?limit=-1&sort=points&cayenneExp=seasonId=20252026"
team_stats = "https://api.nhle.com/stats/rest/en/team/summary?sort=shotsForPerGame&cayenneExp=seasonId=20232024%20and%20gameTypeId=2" #think gametypeid=2 is for reg season and 3 is for playoffs?

play_by_play = "https://api-web.nhle.com/v1/gamecenter/2023020204/play-by-play" #not in a table, but plays are so need code to extract that part of it
shift_chart = "https://api.nhle.com/stats/rest/en/shiftcharts?cayenneExp=gameId=2023020204"
# probs steal the youtubes guys process for putting this together

shift_html = "https://www.nhl.com/scores/htmlreports/20252026/TH020582.HTM"

#2025020693

#reg_season_games.write_excel("_peek.xlsx", autofit=True); __import__("os").startfile("_peek.xlsx")

In [15]:

#====================================================================================
# API Callers
#====================================================================================

# Games
def call_schedule_api(triCode, seasonId) -> pl.DataFrame: 
    url = f"https://api-web.nhle.com/v1/club-schedule-season/{triCode}/{seasonId}"
    response = requests.get(url)
    response.raise_for_status()  
    json_data = response.json()

    df = pl.DataFrame(json_data.get("games", []))

    df = df.select(
            pl.col("id").alias("gameId").cast(pl.Utf8),
            pl.col("season").cast(pl.Utf8),
            pl.col("gameState"),
            pl.col("gameType"),
            pl.col("startTimeUTC"),
            pl.col("venue").struct.field("default").alias("venue"),

            pl.col("homeTeam").struct.field("id").alias("homeTeamId").cast(pl.Utf8),
            pl.col("awayTeam").struct.field("id").alias("awayTeamId").cast(pl.Utf8),
            pl.col("homeTeam").struct.field("abbrev").alias("homeTeamAbbrev").cast(pl.Utf8),
            pl.col("awayTeam").struct.field("abbrev").alias("awayTeamAbbrev").cast(pl.Utf8),

            pl.col("homeTeam").struct.field("score").alias("homeTeamScore"),
            pl.col("awayTeam").struct.field("score").alias("awayTeamScore"),

            pl.col("gameOutcome").struct.field("lastPeriodType").alias("gameOutcome"),

            # pl.col("homeTeam").struct.field("logo").alias("homeTeam_logo"),
            # pl.col("awayTeam").struct.field("logo").alias("awayTeam_logo"),
    )


    df = df.with_columns(
        pl.lit(date.today()).alias("evalDate")
    )

    return df

# Play by Play 
def call_play_by_play_api(gameId: str) -> pl.DataFrame:
    url = f"https://api-web.nhle.com/v1/gamecenter/{gameId}/play-by-play"

    response = requests.get(url)
    response.raise_for_status()
    json_data = response.json()

    df = pl.DataFrame(json_data.get("plays", []))

    away_team_id = (json_data.get("awayTeam") or {}).get("id")
    home_team_id = (json_data.get("homeTeam") or {}).get("id")


    # helper: safely pull optional keys from the nested `details` struct/dict
    # (works even if Polars inferred `details` without that field)
    def details_get(field: str, alias: str | None = None) -> pl.Expr:
        out_name = alias or field
        return pl.col("details").map_elements(
            lambda d: (d.get(field) if d is not None else None)
        ).alias(out_name)

    df = df.select([
        pl.col("eventId").cast(pl.Utf8),
        pl.col("periodDescriptor").struct.field("number").alias("periodNumber"),
        pl.col("timeInPeriod"),
        pl.col("timeRemaining"),
        pl.col("situationCode"),
        pl.col("homeTeamDefendingSide"),
        pl.col("typeCode"),
        pl.col("typeDescKey"),
        pl.col("sortOrder"),

        # details (all treated as optional / blank if missing)
        details_get("eventOwnerTeamId").cast(pl.Utf8),
        details_get("losingPlayerId").cast(pl.Utf8),
        details_get("winningPlayerId").cast(pl.Utf8),
        details_get("xCoord"),
        details_get("yCoord"),
        details_get("zoneCode"),
        details_get("shotType"),
        details_get("shootingPlayerId").cast(pl.Utf8),
        details_get("goalieInNetId").cast(pl.Utf8),
        details_get("awaySOG"),
        details_get("homeSOG"),
        details_get("reason"),
        details_get("playerId").cast(pl.Utf8),
        details_get("blockingPlayerId").cast(pl.Utf8),
        details_get("scoringPlayerId").cast(pl.Utf8),
        details_get("scoringPlayerTotal").cast(pl.Utf8),
        details_get("assist1PlayerId").cast(pl.Utf8),
        details_get("assist1PlayerTotal").cast(pl.Utf8),
        details_get("assist2PlayerId").cast(pl.Utf8),
        details_get("assist2PlayerTotal").cast(pl.Utf8),
        details_get("awayScore"),
        details_get("homeScore"),
        details_get("hittingPlayerId").cast(pl.Utf8),
        details_get("hitteePlayerId").cast(pl.Utf8),
        details_get("secondaryReason"),
        details_get("typeCode", alias="details_typeCode"),  # avoid name clash with top-level typeCode
        details_get("descKey"),
        details_get("duration"),
        details_get("committedByPlayerId").cast(pl.Utf8),
        details_get("drawnByPlayerId").cast(pl.Utf8),
    ])

    df = df.with_columns(
        pl.lit(gameId).alias("gameId").cast(pl.Utf8),
        pl.lit(away_team_id).cast(pl.Utf8).alias("awayTeamId"),
        pl.lit(home_team_id).cast(pl.Utf8).alias("homeTeamId"),
    )

    return df

# Shifts
def call_shift_api(gameId: str) -> pl.DataFrame:

    def normalize_schema(df: pl.DataFrame, schema: dict[str, pl.DataType]) -> pl.DataFrame:
        # add missing cols as nulls with correct dtype
        for col, dtype in schema.items():
            if col not in df.columns:
                df = df.with_columns(pl.lit(None).cast(dtype).alias(col))

        # cast existing cols (strict=False prevents hard failures; bad casts -> null)
        df = df.with_columns([
            pl.col(col).cast(dtype, strict=False)
            for col, dtype in schema.items()
            if col in df.columns
        ])

        return df.select(list(schema.keys()))

    url = f"https://api.nhle.com/stats/rest/en/shiftcharts?cayenneExp=gameId={gameId}"

    r = requests.get(url)
    r.raise_for_status()
    rows = r.json().get("data", [])

    if not rows:
        return empty_df(SHIFT_SCHEMA)

    df = pl.DataFrame(
        rows,
        schema_overrides={"eventDescription": pl.Utf8},  # <-- fixes EVG/PPG/EN
        infer_schema_length=10000,
    )

    SHIFT_SCHEMA = {
        "id": pl.Int64,
        "gameId": pl.Int64,
        "playerId": pl.Int64,
        "period": pl.Int64,
        "shiftNumber": pl.Int64,

        "startTime": pl.Utf8,
        "endTime": pl.Utf8,
        "duration": pl.Utf8,

        # troublemakers: keep as text
        "detailCode": pl.Utf8,
        "typeCode": pl.Utf8,
        "teamId": pl.Utf8,
        "teamAbbrev": pl.Utf8,

        "teamName": pl.Utf8,
        "firstName": pl.Utf8,
        "lastName": pl.Utf8,

        "eventNumber": pl.Int64,
        "eventDescription": pl.Utf8,
        "eventDetails": pl.Utf8,
        "hexValue": pl.Utf8,
    }
    
    df = normalize_schema(df, SHIFT_SCHEMA)

    df = df.select(
                pl.col("gameId").cast(pl.Utf8),
                pl.col("playerId").cast(pl.Utf8),
                pl.col("startTime"),
                pl.col("endTime"),
                pl.col("duration"),
                pl.col("period"),
                pl.col("shiftNumber"),
                pl.col("teamId").cast(pl.Utf8),
                pl.col("eventDescription"),
    )

    return df 

# Players
def call_players_api(season: str) -> pl.DataFrame:
    url = f"https://api.nhle.com/stats/rest/en/skater/summary?limit=-1&sort=points&cayenneExp=seasonId={season}"

    response = requests.get(url)
    response.raise_for_status()
    json_data = response.json()

    df = pl.DataFrame(json_data.get("data", []))


    # filter players to those in the shift list
    df = (
        df
        .with_columns(
            pl.col("playerId").cast(pl.Utf8),
        )
        .select(
            pl.col("playerId").cast(pl.Utf8),
            pl.col("skaterFullName"),
            pl.col("positionCode"),
            pl.col("shootsCatches"),
        )
        .with_columns(
            pl.when(pl.col("positionCode")=='D')
            .then(pl.lit("D"))
            .otherwise(pl.lit("F"))
            .alias("roleCode")
        )
    )

    return df



In [16]:

#====================================================================================
# Simple Helpers
#====================================================================================

def mmss_to_sec(expr: pl.Expr) -> pl.Expr:
    parts = expr.str.split(":")
    return parts.list.get(0).cast(pl.Int64) * 60 + parts.list.get(1).cast(pl.Int64)

def sec_to_period_mmss(abs_sec: pl.Expr) -> tuple[pl.Expr, pl.Expr]:
    period = (abs_sec // (20 * 60) + 1).cast(pl.Int64)
    sec_in_period = (abs_sec % (20 * 60)).cast(pl.Int64)

    mm = (sec_in_period // 60).cast(pl.Int64).cast(pl.Utf8).str.zfill(2)
    ss = (sec_in_period % 60).cast(pl.Int64).cast(pl.Utf8).str.zfill(2)

    mmss = mm + pl.lit(":") + ss
    return period, mmss

def time_mmss_to_seconds(expr: pl.Expr) -> pl.Expr:
    """
    Convert 'MM:SS' string column to integer seconds.
    """
    mins = expr.str.slice(0, 2).cast(pl.Int64)
    secs = expr.str.slice(3, 2).cast(pl.Int64)
    return (mins * 60 + secs)

def get_season_id(gameId: str) -> str:
    start_year = int(gameId[:4])
    return f"{start_year}{start_year + 1}"

def pull_live_game(triCode, seasonId) -> pl.DataFrame:

    df = call_schedule_api(triCode,seasonId)

    df = df.filter(pl.col("gameState") == "LIVE")

    return df



In [17]:
#====================================================================================
# Table/Complex Helpers
#====================================================================================

def single_game_shifts(gameId: str, playersAll: pl.DataFrame | None = None) -> pl.DataFrame:
    seasonId = get_season_id(gameId)

    if playersAll is None:
        playersAll = (
            call_players_api(seasonId)
            .with_columns(pl.col("playerId").cast(pl.Utf8))
        )

    shifts = call_shift_api(gameId)

    shifts = (
        shifts
        .with_columns(
            pl.col("startTime").rank("dense").over(["gameId", "playerId", "period"]).alias("shift"),
            (pl.col("period") + pl.col("shiftNumber") / 100).alias("shiftId"),
            pl.col("playerId").cast(pl.Utf8),
        )
        .join(playersAll, on="playerId", how="left")
        .filter(pl.col("skaterFullName") != "")
    )

    out = (
        shifts
        .with_columns(
            pl.col("gameId").cast(pl.Utf8),
            pl.col("playerId").cast(pl.Utf8),
            pl.col("teamId").cast(pl.Utf8),

            pl.col("period").cast(pl.Int64),
            pl.col("shiftNumber").cast(pl.Int64),

            startSec=mmss_to_sec(pl.col("startTime")).cast(pl.Int64),
            endSec=mmss_to_sec(pl.col("endTime")).cast(pl.Int64),
        )
        .with_columns(
            absStart=((pl.col("period") - 1) * 20 * 60 + pl.col("startSec")).cast(pl.Int64),
            absEnd=((pl.col("period") - 1) * 20 * 60 + pl.col("endSec")).cast(pl.Int64),
        )
        .filter(pl.col("absEnd") > pl.col("absStart"))
        .with_columns(
            durationSec=(pl.col("absEnd") - pl.col("absStart")).cast(pl.Int64)
        )
        .drop([c for c in ["duration", "shiftId", "eventDescription", "shift"] if c in shifts.columns])
    )

    keepCols = [
        "gameId", "playerId", "teamId",
        "period", "shiftNumber",
        "startTime", "endTime",
        "startSec", "endSec",
        "absStart", "absEnd",
        "durationSec",
        "roleCode",
    ]

    return out.select([c for c in keepCols if c in out.columns])


In [ ]:

#====================================================================================
# Database Spine Tables
#====================================================================================

# filtering only to games that have happened
def db_games(schedTables, gameType) -> pl.DataFrame:
    dfs = []

    for row in schedTables.iter_rows(named=True):
        df = call_schedule_api(row['triCode'],row['seasonId'])
        dfs.append(df)

    sched_output = pl.concat(dfs, how="diagonal")

    if gameType > -1:
        sched_output = sched_output.filter(pl.col("gameType") == gameType)

    # de-duplicate games pulled from multiple teams' schedules
    games = sched_output.unique(subset=["gameId"])

    games = games.filter(pl.col("gameState") != "FUT")

    return games

def db_teams(url: str = "https://api-web.nhle.com/v1/standings/now") -> pl.DataFrame:
    """
    Pull NHL standings snapshot and return a lightweight team lookup table with:
      - conferenceAbbrev
      - divisionAbbrev
      - teamName
      - teamAbbrev
      - teamLogo
    """
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    standings = data.get("standings", [])
    df = pl.DataFrame(standings)

    out = (
        df.select([
            pl.col("conferenceAbbrev"),
            pl.col("divisionAbbrev"),
            pl.col("teamName").struct.field("default").alias("teamName"),
            pl.col("teamAbbrev").struct.field("default").alias("teamAbbrev"),
            pl.col("teamLogo"),
        ])
        .unique(subset=["teamAbbrev"])  # should already be unique, but safe
        .sort(["conferenceAbbrev", "divisionAbbrev", "teamAbbrev"])
    )

    # pull games from last full season to get teamTds
    season_table = pl.DataFrame({
            "triCode": ["MIN"],
            "seasonId": [20242025], 
        }
    )
    games = db_games(season_table, -1)

    teams_from_games = (
        pl.concat([
            games.select([pl.col("homeTeamId").alias("teamId"), pl.col("homeTeamAbbrev").alias("teamAbbrev")]),
            games.select([pl.col("awayTeamId").alias("teamId"), pl.col("awayTeamAbbrev").alias("teamAbbrev")]),
        ])
        .unique(subset=["teamAbbrev"])
    )
    
    out = (
        out
        .join(teams_from_games, on="teamAbbrev", how="left")
        .select("teamId","teamAbbrev","teamName","conferenceAbbrev","divisionAbbrev","teamLogo")
    )

    return out

def db_pbp(games: pl.DataFrame) -> pl.DataFrame:
    gameIds = (
        games
        .select(pl.col("gameId"))
        .unique()
        .get_column("gameId")
        .to_list()
    )

    dfs = []
    for gid in gameIds:
        try:
            pbp = call_play_by_play_api(gid) 
            dfs.append(pbp.with_columns(pl.col("gameId").cast(pl.Utf8)))
        except Exception as e:
            print(f"Failed game {gid}: {e}")

    if not dfs:
        return pl.DataFrame()

    return pl.concat(dfs, how="diagonal")

def db_shifts(games) -> pl.DataFrame:
    
    """
    Calls dbShifts(gameId) for each unique gameId in dfGames and concatenates results.
    Caches playersAll per seasonId to avoid repeated player API calls.
    Expects dfGames to contain 'gameId'.
    """
    gameIds = (
        games
        .select(pl.col("gameId").cast(pl.Utf8).unique())
        .get_column("gameId")
        .to_list()
    )

    playersCache: dict[str, pl.DataFrame] = {}
    dfs: list[pl.DataFrame] = []

    for gameId in gameIds:
        try:
            seasonId = get_season_id(gameId)

            if seasonId not in playersCache:
                playersCache[seasonId] = (
                    call_players_api(seasonId)
                    .with_columns(pl.col("playerId").cast(pl.Utf8))
                )

            dfs.append(single_game_shifts(gameId, playersAll=playersCache[seasonId]))

        except Exception as e:
            print(f"single_game_shifts failed for gameId={gameId}: {e}")

    if not dfs:
        return pl.DataFrame()

    return pl.concat(dfs, how="diagonal")

def db_eventOnIce(pbp: pl.DataFrame, shifts: pl.DataFrame) -> pl.DataFrame:
    """
    Build normalized eventOnIce bridge table.

    Input:
      pbp: event-grain table containing at least
           ["gameId", "eventId", "periodNumber", "timeInPeriod"]
      shifts: shift-grain table containing at least
           ["gameId", "playerId", "teamId", "absStart", "absEnd"]
           (and optionally "roleCode" to drop goalies)

    Output (grain: gameId-eventId-playerId):
      ["gameId", "eventId", "playerId", "teamId"]
    """

    # --- event seconds (absolute) ---
    pbpEvents = (
        pbp
        .with_columns(
            pl.col("gameId").cast(pl.Utf8),
            pl.col("eventId").cast(pl.Utf8),
            eventSec=((pl.col("periodNumber") - 1) * 20 * 60 + mmss_to_sec(pl.col("timeInPeriod"))).cast(pl.Int64),
        )
        .select(["gameId", "eventId", "eventSec"])
        .unique(subset=["gameId", "eventId"])
    )

    # --- shift intervals (absolute) ---
    shiftsClean = (
        shifts
        .with_columns(
            pl.col("gameId").cast(pl.Utf8),
            pl.col("playerId").cast(pl.Utf8),
            pl.col("teamId").cast(pl.Utf8),
            pl.col("absStart").cast(pl.Int64),
            pl.col("absEnd").cast(pl.Int64),
        )
        .filter(pl.col("absEnd") > pl.col("absStart"))
        .select(["gameId", "teamId", "playerId", "absStart", "absEnd"] + (["roleCode"] if "roleCode" in shifts.columns else []))
    )

    # Drop goalies if roleCode is available
    if "roleCode" in shiftsClean.columns:
        shiftsClean = shiftsClean.filter(pl.col("roleCode") != "G").drop("roleCode")

    # --- interval match: absStart <= eventSec < absEnd ---
    eventOnIce = (
        pbpEvents
        .join(shiftsClean, on="gameId", how="inner")
        .filter((pl.col("absStart") <= pl.col("eventSec")) & (pl.col("eventSec") < pl.col("absEnd")))
        .select(["gameId", "eventId", "playerId", "teamId"])
        .unique()
        .sort(["gameId", "eventId", "teamId", "playerId"])
    )

    return eventOnIce

def db_eventStrength(eventOnIce: pl.DataFrame, pbp: pl.DataFrame,) -> pl.DataFrame:
    """
    Build eventStrength table.

    Inputs:
      eventOnIce: columns ["gameId","eventId","playerId","teamId"] (goalies excluded upstream)
      pbp: must contain ["gameId","eventId","homeTeamId","awayTeamId"]

    Output grain: (gameId, eventId)
    Columns:
      gameId, eventId,
      homeSkaters, awaySkaters,
      strengthState,
      is5v5, is5v4, is4v5, is4v4, is3v3,
      is6v5, is5v6, is6v4, is4v6, is6v6,
      homeEmptyNet, awayEmptyNet, isEmptyNet
    """

    eventTeams = (
        pbp
        .select(["gameId", "eventId", "homeTeamId", "awayTeamId"])
        .with_columns(
            pl.col("gameId").cast(pl.Utf8),
            pl.col("eventId").cast(pl.Utf8),
            pl.col("homeTeamId").cast(pl.Utf8),
            pl.col("awayTeamId").cast(pl.Utf8),
        )
        .unique(subset=["gameId", "eventId"])
    )

    counts = (
        eventOnIce
        .select(["gameId", "eventId", "teamId", "playerId"])
        .with_columns(
            pl.col("gameId").cast(pl.Utf8),
            pl.col("eventId").cast(pl.Utf8),
            pl.col("teamId").cast(pl.Utf8),
            pl.col("playerId").cast(pl.Utf8),
        )
        .unique(subset=["gameId", "eventId", "teamId", "playerId"])
        .group_by(["gameId", "eventId", "teamId"])
        .agg(homeAwaySkaters=pl.n_unique("playerId").cast(pl.Int64).alias("skaters"))
        .rename({"homeAwaySkaters": "skaters"})
    )

    # attach home/away + pivot to homeSkaters/awaySkaters
    out = (
        eventTeams
        .join(counts, on=["gameId", "eventId"], how="left")
        .with_columns(
            side=pl.when(pl.col("teamId") == pl.col("homeTeamId"))
                  .then(pl.lit("home"))
                  .when(pl.col("teamId") == pl.col("awayTeamId"))
                  .then(pl.lit("away"))
                  .otherwise(pl.lit(None)),
        )
        .filter(pl.col("side").is_not_null())
        .select(["gameId", "eventId", "side", "skaters"])
        .pivot(
            values="skaters",
            index=["gameId", "eventId"],
            on="side",
        )
        .rename({"home": "homeSkaters", "away": "awaySkaters"})
        .with_columns(
            pl.col("homeSkaters").fill_null(0).cast(pl.Int64),
            pl.col("awaySkaters").fill_null(0).cast(pl.Int64),
        )
        .with_columns(
            strengthState=pl.format("{}v{}", pl.col("homeSkaters"), pl.col("awaySkaters")),

            # common even/PP states
            is5v5=((pl.col("homeSkaters") == 5) & (pl.col("awaySkaters") == 5)).cast(pl.Int8),
            is5v4=((pl.col("homeSkaters") == 5) & (pl.col("awaySkaters") == 4)).cast(pl.Int8),
            is4v5=((pl.col("homeSkaters") == 4) & (pl.col("awaySkaters") == 5)).cast(pl.Int8),
            is4v4=((pl.col("homeSkaters") == 4) & (pl.col("awaySkaters") == 4)).cast(pl.Int8),
            is3v3=((pl.col("homeSkaters") == 3) & (pl.col("awaySkaters") == 3)).cast(pl.Int8),

            # pulled-goalie / extra-attacker states
            is6v5=((pl.col("homeSkaters") == 6) & (pl.col("awaySkaters") == 5)).cast(pl.Int8),
            is5v6=((pl.col("homeSkaters") == 5) & (pl.col("awaySkaters") == 6)).cast(pl.Int8),
            is6v4=((pl.col("homeSkaters") == 6) & (pl.col("awaySkaters") == 4)).cast(pl.Int8),
            is4v6=((pl.col("homeSkaters") == 4) & (pl.col("awaySkaters") == 6)).cast(pl.Int8),
            is6v6=((pl.col("homeSkaters") == 6) & (pl.col("awaySkaters") == 6)).cast(pl.Int8),

            # empty net flags (approximation via skater count)
            homeEmptyNet=(pl.col("homeSkaters") > 5).cast(pl.Int8),
            awayEmptyNet=(pl.col("awaySkaters") > 5).cast(pl.Int8),
        )
        .with_columns(
            isEmptyNet=(pl.col("homeEmptyNet") | pl.col("awayEmptyNet")).cast(pl.Int8)
        )
        .sort(["gameId", "eventId"])
    )

    return out

def db_eventTeamFlags(pbp: pl.DataFrame, shifts: pl.DataFrame) -> pl.DataFrame:
    """
    Build eventTeamFlags table (NST-style attribution).

    Grain: (gameId, eventId, teamId)

    Output columns:
      gameId, eventId, teamId,
      GF, GA, CF, CA, FF, FA

    Attribution:
      - FOR team for shot attempts/goals = shooter's team
        (shootingPlayerId -> shifts playerId/teamId mapping)
      - fallback to eventOwnerTeamId when shooter team missing
    """

    shotEvents = ["shot-on-goal", "goal", "missed-shot", "blocked-shot"]
    fenwickEvents = ["shot-on-goal", "goal", "missed-shot"]

    # Map playerId -> teamId (within game) from shifts
    playerTeam = (
        shifts
        .select(["gameId", "playerId", "teamId"])
        .with_columns(
            pl.col("gameId").cast(pl.Utf8),
            pl.col("playerId").cast(pl.Utf8),
            pl.col("teamId").cast(pl.Utf8),
        )
        .unique(subset=["gameId", "playerId"])
    )

    # Minimal pbp view with shooter team attached
    pbp2 = (
        pbp
        .select(["gameId", "eventId", "typeDescKey", "eventOwnerTeamId", "shootingPlayerId", "homeTeamId", "awayTeamId"])
        .with_columns(
            pl.col("gameId").cast(pl.Utf8),
            pl.col("eventId").cast(pl.Utf8),
            pl.col("eventOwnerTeamId").cast(pl.Utf8),
            pl.col("shootingPlayerId").cast(pl.Utf8),
            pl.col("homeTeamId").cast(pl.Utf8),
            pl.col("awayTeamId").cast(pl.Utf8),
        )
        .join(
            playerTeam.rename({"playerId": "shootingPlayerId", "teamId": "shooterTeamId"}),
            on=["gameId", "shootingPlayerId"],
            how="left",
        )
        .with_columns(
            shotForTeamId=pl.coalesce([pl.col("shooterTeamId"), pl.col("eventOwnerTeamId")])
        )
        .select(["gameId", "eventId", "typeDescKey", "shotForTeamId", "homeTeamId", "awayTeamId"])
    )

    # Build two rows per event: home+away
    homeRows = pbp2.select(["gameId", "eventId", "typeDescKey", "shotForTeamId", "homeTeamId"]).rename({"homeTeamId": "teamId"})
    awayRows = pbp2.select(["gameId", "eventId", "typeDescKey", "shotForTeamId", "awayTeamId"]).rename({"awayTeamId": "teamId"})

    et = pl.concat([homeRows, awayRows], how="vertical")

    et = (
        et
        .with_columns(
            isGoal=(pl.col("typeDescKey") == "goal").cast(pl.Int8),
            isCorsiEvent=pl.col("typeDescKey").is_in(shotEvents).cast(pl.Int8),
            isFenwickEvent=pl.col("typeDescKey").is_in(fenwickEvents).cast(pl.Int8),
            isFor=(pl.col("teamId") == pl.col("shotForTeamId")).cast(pl.Int8),
        )
        .with_columns(
            GF=(pl.col("isGoal") & (pl.col("isFor") == 1)).cast(pl.Int8),
            GA=(pl.col("isGoal") & (pl.col("isFor") == 0)).cast(pl.Int8),
            CF=(pl.col("isCorsiEvent") & (pl.col("isFor") == 1)).cast(pl.Int8),
            CA=(pl.col("isCorsiEvent") & (pl.col("isFor") == 0)).cast(pl.Int8),
            FF=(pl.col("isFenwickEvent") & (pl.col("isFor") == 1)).cast(pl.Int8),
            FA=(pl.col("isFenwickEvent") & (pl.col("isFor") == 0)).cast(pl.Int8),
        )
        .select(["gameId", "eventId", "teamId", "GF", "GA", "CF", "CA", "FF", "FA"])
    )

    return et


In [ ]:
################# DATABASE ################# 

season_table = pl.DataFrame(
    {
        "triCode": ["MIN"],
        "seasonId": [20242025], 
    }
)




#pull games for the teams and seasons
DF_GAMES = db_games(season_table, -1) # 1 pre season // 2 regular season // 3 playoffs // 4 all stars

DF_TEAMS = db_teams()

#DF_PBPS = db_pbp(DF_GAMES)

DF_SHIFTS = db_shifts(DF_GAMES)

DF_EVENT_ON_ICE = db_eventOnIce(DF_PBPS, DF_SHIFTS)

DF_EVENT_STRENGTH = db_eventStrength(DF_EVENT_ON_ICE, DF_PBPS)

DF_EVENT_TEAM_FLAGS = db_eventTeamFlags(DF_PBPS, DF_SHIFTS)

In [20]:

#pull games for the teams and seasons
live_game = pull_live_game("MTL",current_season)
#live_game.write_excel("_games.xlsx", autofit=True); __import__("os").startfile("_games.xlsx")

pbp = db_pbp(live_game)
pbp.write_excel("_pbp.xlsx", autofit=True); __import__("os").startfile("_pbp.xlsx")

shifts = db_shifts(live_game)
shifts.write_excel("_shifts.xlsx", autofit=True); __import__("os").startfile("_shifts.xlsx")

events_on_ice = db_eventOnIce(pbp, shifts)
events_on_ice.write_excel("_events_on_ice.xlsx", autofit=True); __import__("os").startfile("_events_on_ice.xlsx")

event_strength = db_eventStrength(events_on_ice, pbp)
event_strength.write_excel("_event_strength.xlsx", autofit=True); __import__("os").startfile("_event_strength.xlsx")

event_team_flags = db_eventTeamFlags(pbp, shifts)
event_team_flags.write_excel("_event_team_flags.xlsx", autofit=True); __import__("os").startfile("_event_team_flags.xlsx")

In [ ]:
from __future__ import annotations

import re
from typing import Dict, Optional, Tuple
import polars as pl
import requests
from datetime import date
from bs4 import BeautifulSoup

# vars.py
last_season = 20242025
current_season = 20252026
overlap_duration = 35 #seconds

# helpers.py
def parse_shift_report_html(html: str, *, debug: bool = False) -> pl.DataFrame:
    soup = BeautifulSoup(html, "html.parser")

    teamName = soup.find("td", class_=["teamHeading"]).text.strip().title()

    rows = []
    current_player = None

    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        if not tds:
            continue

        # 1) Player header row: <td class="playerHeading ...">6 LARSSON, ADAM</td>
        player_td = tr.find("td", class_=lambda c: c and "playerHeading" in c)
        if player_td is not None:
            current_player = player_td.get_text(" ", strip=True).title()
            if debug:
                print("PLAYER:", current_player)
            continue

        # 2) Skip the column header row (Shift #, Per, Start of Shift, ...)
        first_txt = tds[0].get_text(" ", strip=True)
        if first_txt.lower().startswith("shift"):
            continue

        # 3) Shift rows look like 6 tds:
        # [shift#, per, start, end, duration, event]
        if len(tds) == 6 and current_player:
            shift_txt = tds[0].get_text(" ", strip=True)
            per_txt = tds[1].get_text(" ", strip=True)

            # must be numeric shift# and period
            if shift_txt.isdigit() and per_txt.isdigit():
                start_txt = tds[2].get_text(" ", strip=True)
                end_txt = tds[3].get_text(" ", strip=True)
                dur_txt = tds[4].get_text(" ", strip=True)

                # event cell sometimes is &nbsp; (becomes '' after strip)
                event_txt = tds[5].get_text(" ", strip=True) if len(tds) > 5 else ""
                event_txt = event_txt if event_txt else None

                rows.append(
                    {
                        "player": current_player,
                        "shiftNumber": int(shift_txt),
                        "period": int(per_txt),
                        "startOfShift": start_txt,   
                        "endOfShift": end_txt,       
                    }
                )

    df = pl.DataFrame(rows)

    df = df.with_columns(pl.lit(teamName).alias("teamName"))

    return df 
def get_season_id(gameId: str) -> str:
    start_year = int(gameId[:4])
    return f"{start_year}{start_year + 1}"
def mmss_to_sec(expr: pl.Expr) -> pl.Expr:
    parts = expr.str.split(":")
    return parts.list.get(0).cast(pl.Int64) * 60 + parts.list.get(1).cast(pl.Int64)

# api.py
def call_schedule_api(triCode, seasonId) -> pl.DataFrame: 
    url = f"https://api-web.nhle.com/v1/club-schedule-season/{triCode}/{seasonId}"
    response = requests.get(url)
    response.raise_for_status()  
    json_data = response.json()

    df = pl.DataFrame(json_data.get("games", []))

    df = df.select(
            pl.col("id").alias("gameId").cast(pl.Utf8),
            pl.col("season").cast(pl.Utf8),
            pl.col("gameState"),
            pl.col("gameType"),
            pl.col("startTimeUTC"),
            pl.col("venue").struct.field("default").alias("venue"),

            pl.col("homeTeam").struct.field("id").alias("homeTeamId").cast(pl.Utf8),
            pl.col("awayTeam").struct.field("id").alias("awayTeamId").cast(pl.Utf8),
            pl.col("homeTeam").struct.field("abbrev").alias("homeTeamAbbrev").cast(pl.Utf8),
            pl.col("awayTeam").struct.field("abbrev").alias("awayTeamAbbrev").cast(pl.Utf8),

            pl.col("homeTeam").struct.field("score").alias("homeTeamScore"),
            pl.col("awayTeam").struct.field("score").alias("awayTeamScore"),

            pl.col("gameOutcome").struct.field("lastPeriodType").alias("gameOutcome"),

            # pl.col("homeTeam").struct.field("logo").alias("homeTeam_logo"),
            # pl.col("awayTeam").struct.field("logo").alias("awayTeam_logo"),
    )


    df = df.with_columns(
        pl.lit(date.today()).alias("evalDate")
    )

    return df
def call_players_api(season: str) -> pl.DataFrame:
    url = f"https://api.nhle.com/stats/rest/en/skater/summary?limit=-1&sort=points&cayenneExp=seasonId={season}"


    response = requests.get(url)
    response.raise_for_status()
    json_data = response.json()

    df = pl.DataFrame(json_data.get("data", []))

    return df
def call_goalies_api(season: str) -> pl.DataFrame:
    url = f"https://api.nhle.com/stats/rest/en/goalie/summary?limit=-1&sort=wins&cayenneExp=seasonId={season}"


    response = requests.get(url)
    response.raise_for_status()
    json_data = response.json()

    df = pl.DataFrame(json_data.get("data", []))

    return df

# database_builder.py
def db_players(seasonId) -> pl.DataFrame:
    players = call_players_api(seasonId)
    goalies = call_goalies_api(seasonId)

    # filter players to those in the shift list
    players = (
        players
        .with_columns(
            pl.col("playerId").cast(pl.Utf8),
        )
        .select(
            pl.col("playerId").cast(pl.Utf8),
            pl.col("skaterFullName"),
            pl.col("positionCode"),
            pl.col("shootsCatches"),
        )
        .with_columns(
            pl.when(pl.col("positionCode")=='D')
            .then(pl.lit("D"))
            .otherwise(pl.lit("F"))
            .alias("roleCode")
        )
    )
    goalies = (
        goalies
        .with_columns(
            pl.col("playerId").cast(pl.Utf8),
        )
        .select(
            pl.col("playerId").cast(pl.Utf8),
            pl.col("goalieFullName").alias("skaterFullName"),
            pl.col("shootsCatches"),
        )
        .with_columns(
            pl.lit("G").alias("positionCode"),
            pl.lit("G").alias("roleCode"),
        )
        .select("playerId", "skaterFullName", "positionCode", "shootsCatches", "roleCode")
    )

    return pl.concat([players, goalies], how="vertical")
def db_games(schedTables, seasonId, gameType) -> pl.DataFrame:
    dfs = []

    for row in schedTables.iter_rows(named=True):
        df = call_schedule_api(row['triCode'],seasonId)
        dfs.append(df)

    sched_output = pl.concat(dfs, how="diagonal")

    if gameType > -1:
        sched_output = sched_output.filter(pl.col("gameType") == gameType)

    # de-duplicate games pulled from multiple teams' schedules
    games = sched_output.unique(subset=["gameId"])

    games = (
        games
        .with_columns(
            pl.col("startTimeUTC")
            .str.slice(0, 10)
            .alias("gameDate")
        )
        .filter(pl.col("gameState") != "FUT")
    )
    return games
def db_teams(url: str = "https://api-web.nhle.com/v1/standings/now") -> pl.DataFrame:
    """
    Pull NHL standings snapshot and return a lightweight team lookup table with:
      - conferenceAbbrev
      - divisionAbbrev
      - teamName
      - teamAbbrev
      - teamLogo
    """
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    standings = data.get("standings", [])
    df = pl.DataFrame(standings)

    out = (
        df.select([
            pl.col("conferenceAbbrev"),
            pl.col("divisionAbbrev"),
            pl.col("teamName").struct.field("default").alias("teamName"),
            pl.col("teamAbbrev").struct.field("default").alias("teamAbbrev"),
            pl.col("teamLogo"),
        ])
        .unique(subset=["teamAbbrev"])  # should already be unique, but safe
        .sort(["conferenceAbbrev", "divisionAbbrev", "teamAbbrev"])
    )

    # pull games from last full season to get teamTds
    season_table = pl.DataFrame({
            "triCode": ["MIN"],
        }
    )
    games = db_games(season_table, last_season, -1)

    teams_from_games = (
        pl.concat([
            games.select([pl.col("homeTeamId").alias("teamId"), pl.col("homeTeamAbbrev").alias("teamAbbrev")]),
            games.select([pl.col("awayTeamId").alias("teamId"), pl.col("awayTeamAbbrev").alias("teamAbbrev")]),
        ])
        .unique(subset=["teamAbbrev"])
    )
    
    out = (
        out
        .join(teams_from_games, on="teamAbbrev", how="left")
        .select("teamId","teamAbbrev","teamName","conferenceAbbrev","divisionAbbrev","teamLogo")
    )

    return out



def db_shift_html(
    gameId: str,
    players: pl.DataFrame,
    teams: pl.DataFrame,
    *,
    timeout_sec: int = 20,
    debug: bool = False,
) -> pl.DataFrame:
    

    df = pl.DataFrame(
        schema={
            "player": pl.Utf8,
            "shiftNumber": pl.Int64,
            "period": pl.Int64,
            "startOfShift": pl.Utf8,
            "endOfShift": pl.Utf8,
            "teamName": pl.Utf8,
        }
    )


    html_home = call_shift_html(gameId,"H")
    df_home = parse_shift_report_html(html_home, debug=False)

    html_away = call_shift_html(gameId,"V")
    df_away = parse_shift_report_html(html_away, debug=False)

    df_combined = pl.concat([df_home, df_away], how="vertical")
    
    df_combined = (
        df_combined
        .with_columns(
            pl.lit(gameId).alias("gameId"),
            (pl.col("player").str.split(by=" ").list.get(2)+" "+pl.col("player").str.split(by=" ").list.get(1)).str.replace(",", "").alias("skaterFullName"),
            pl.col("startOfShift").str.split(by=" ").list.get(0).alias("startTime"),
            pl.col("endOfShift").str.split(by=" ").list.get(0).alias("endTime"),
        )
        .with_columns(
            startSec=mmss_to_sec(pl.col("startTime")).cast(pl.Int64),
            endSec=mmss_to_sec(pl.col("endTime")).cast(pl.Int64),
        )
        .with_columns(
            absStart=((pl.col("period") - 1) * 20 * 60 + pl.col("startSec")).cast(pl.Int64),
            absEnd=((pl.col("period") - 1) * 20 * 60 + pl.col("endSec")).cast(pl.Int64),
        )
        .join(players.select("skaterFullName", "playerId", "roleCode"), on="skaterFullName", how="left")
        .join(teams.select("teamName", "teamId"), on="teamName", how="left")
    )

    df_combined = (df_combined.select("gameId", "playerId", "teamId", "period", "shiftNumber", "startTime", "endTime", "startSec", "endSec", "absStart", "absEnd", "roleCode"))

    return df_combined

gameId = "2025020693"
players = db_players(get_season_id(gameId))
teams = db_teams()
x=db_shift_html(gameId, players, teams)
#x.write_excel("_games.xlsx", autofit=True); __import__("os").startfile("_games.xlsx")
x



gameId,playerId,teamId,period,shiftNumber,startTime,endTime,startSec,endSec,absStart,absEnd,roleCode
str,str,str,i64,i64,str,str,i64,i64,i64,i64,str
"""2025020693""","""8476457""","""55""",1,1,"""0:00""","""0:40""",0,40,0,40,"""D"""
"""2025020693""","""8476457""","""55""",1,2,"""2:20""","""3:26""",140,206,140,206,"""D"""
"""2025020693""","""8476457""","""55""",1,3,"""5:00""","""5:45""",300,345,300,345,"""D"""
"""2025020693""","""8476457""","""55""",1,4,"""6:30""","""7:40""",390,460,390,460,"""D"""
"""2025020693""","""8476457""","""55""",1,5,"""9:32""","""10:06""",572,606,572,606,"""D"""
…,…,…,…,…,…,…,…,…,…,…,…
"""2025020693""","""8478864""","""30""",3,16,"""4:44""","""6:33""",284,393,2684,2793,"""F"""
"""2025020693""","""8478864""","""30""",3,17,"""9:07""","""10:22""",547,622,2947,3022,"""F"""
"""2025020693""","""8478864""","""30""",3,18,"""11:21""","""12:54""",681,774,3081,3174,"""F"""
